In [1]:
import os
import sys

# Replace these paths with the actual path to Java on your machine
os.environ["JAVA_HOME"] = "/home/jbisson/.sdkman/candidates/java/17.0.15-librca/"
os.environ["SPARK_LOCAL_IP"]= "127.0.0.1"

# Python executable path (helps Spark find uv's virtual env python)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
from pyspark.sql import SparkSession


# Replace these with your actual Apache Polaris details
POLARIS_URI = "http://localhost:8181/api/catalog"   # Your Polaris REST endpoint
CLIENT_ID = "root"                                  # The Principal Client ID
CLIENT_SECRET = "s3cr3t"                            # The Principal Client Secret
CATALOG_NAME = "test_catalog"                       # The catalog name defined in Polaris



# Create a local Spark session
spark = ( 
    SparkSession.builder
    .appName("LocalPySparkSession")
    .master("local[*]")
    # 1 Download the iceberg spark runtime dependencies
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.10.1")
    # 2. Enable Iceberg Extensions
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # 3. Define the 'test_catalog' as a Polaris catalog running locally
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", POLARIS_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", CATALOG_NAME)
    # 3. Configure Authentication (Username and Password)
    # The standard Iceberg REST convention packs them into a 'username:password' credential string
    .config(f"spark.sql.catalog.{CATALOG_NAME}.scope", "PRINCIPAL_ROLE:ALL")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.credential", f"{CLIENT_ID}:{CLIENT_SECRET}")
        # 4. Set test_catalog as the default catalog for this session
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    .getOrCreate()
)
# Verify the session is working
print(f"Spark Version: {spark.version}")
print(f"Spark App Name: {spark.sparkContext.appName}")

# Quick sanity check: Create a tiny DataFrame
data = [("Alice", 25), ("Bob", 30), ("Charlie", 35)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, schema=columns)
df.show()

:: loading settings :: url = jar:file:/home/jbisson/github/polaris/runtime/polaris-self-serve/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jbisson/.ivy2/cache
The jars for the packages stored in: /home/jbisson/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-66292203-3312-488a-8042-4f28288b4232;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.1 in central
:: resolution report :: resolve 315ms :: artifacts dl 10ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apac

Spark Version: 3.5.6
Spark App Name: LocalPySparkSession


26/06/03 08:25:51 WARN AuthManagers: Inferring rest.auth.type=oauth2 since property credential was provided. Please explicitly set rest.auth.type to avoid this warning.
26/06/03 08:25:52 WARN OAuth2Manager: Iceberg REST client is missing the OAuth2 server URI configuration and defaults to http://localhost:8181/api/catalog/v1/oauth/tokens. This automatic fallback will be removed in a future Iceberg release. It is recommended to configure the OAuth2 endpoint using the 'oauth2-server-uri' property to be prepared. This warning will disappear if the OAuth2 endpoint is explicitly configured. See https://github.com/apache/iceberg/issues/10537


+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



In [11]:
spark.sql('SHOW NAMESPACES IN test_catalog').show(truncate=False)
spark.sql('DESCRIBE NAMESPACE EXTENDED test_catalog.test_namespace').show(truncate=False)

+--------------+
|namespace     |
+--------------+
|test_namespace|
+--------------+

+--------------+--------------------------------------------------------------------+
|info_name     |info_value                                                          |
+--------------+--------------------------------------------------------------------+
|Catalog Name  |test_catalog                                                        |
|Namespace Name|test_namespace                                                      |
|Location      |file:///tmp/test_catalog/test_namespace/                            |
|Owner         |root                                                                |
|Properties    |((owner_created_at,2026-06-03T12:39:14.628909660Z), (owner_id,root))|
+--------------+--------------------------------------------------------------------+



26/06/03 11:10:53 WARN Tasks: Retrying task after failure: sleepTimeMs=106 Error occurred while processing POST request
org.apache.iceberg.exceptions.RESTException: Error occurred while processing POST request
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:359)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:297)
	at org.apache.iceberg.rest.BaseHTTPClient.postForm(BaseHTTPClient.java:136)
	at org.apache.iceberg.rest.auth.OAuth2Util.refreshToken(OAuth2Util.java:174)
	at org.apache.iceberg.rest.auth.OAuth2Util$AuthSession.refreshExpiredToken(OAuth2Util.java:638)
	at org.apache.iceberg.rest.auth.OAuth2Util$AuthSession.refreshCurrentToken(OAuth2Util.java:623)
	at org.apache.iceberg.rest.auth.OAuth2Util$AuthSession.lambda$refresh$1(OAuth2Util.java:596)
	at org.apache.iceberg.util.Tasks$Builder.runTaskWithRetry(Tasks.java:413)
	at org.apache.iceberg.util.Tasks$Builder.runSingleThreaded(Tasks.java:219)
	at org.apache.iceberg.util.Tasks$Builder.run(Tasks.java:203

In [8]:
# Quick sanity check: Create a tiny DataFrame
data = [("Alice", 25), ("Bob", 30), ("Charlie", 35)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, schema=columns)
# Define the full table path
table_path = "test_catalog.test_namespace.test_table"

# Write and create the table
df.writeTo(table_path).create()

26/06/03 08:39:35 WARN RESTMetricsReporter: Failed to report metrics to REST endpoint v1/test_catalog/namespaces/test_namespace/tables/test_table/metrics
org.apache.iceberg.exceptions.RESTException: Unable to process: Table does not exist: test_namespace.test_table
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:250)
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:214)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:240)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:336)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:297)
	at org.apache.iceberg.rest.BaseHTTPClient.post(BaseHTTPClient.java:100)
	at org.apache.iceberg.rest.RESTClient.post(RESTClient.java:126)
	at org.apache.iceberg.rest.RESTMetricsReporter.lambda$report$1(RESTMetricsReporter.java:69)
	at org.apache.iceberg.util.Tasks$Builder.runTaskWithRetry(Tasks.java:413)
	at org.apache.iceberg.util.Tasks

In [9]:
spark.sql('SELECT * FROM test_catalog.test_namespace.test_table').show(truncate=False)

26/06/03 08:40:52 WARN RESTMetricsReporter: Failed to report metrics to REST endpoint v1/test_catalog/namespaces/test_namespace/tables/test_table/metrics
org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'root' with activated PrincipalRoles '[service_admin]' and activated grants via '[service_admin, catalog_admin]' is not authorized for op REPORT_READ_METRICS
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:214)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:240)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:336)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:297)
	at org.apache.iceberg.rest.BaseHTTPClient.post(BaseHTTPClient.java:100)
	at org.apache.iceberg.rest.RESTClient.post(RESTClient.java:126)
	at org.apache.iceberg.rest.RESTMetricsReporter.lambda$report$1(RESTMetricsReporter.

+-------+---+
|Name   |Age|
+-------+---+
|Alice  |25 |
|Bob    |30 |
|Charlie|35 |
+-------+---+

